# Competitive CNN: khảo sát kernel thời gian k = 1…9

Mỗi kernel có kích thước **512 × k**, toàn bộ Conv có **128 kernels**, chia hai nhóm 64/64. Giữ Competitive có GELU và Top-K video pooling. Chỉ thay k giữa các lượt; không chạy song song.

Train from scratch, seed model và DataLoader 234, 10 epoch, LR 5e-6, giữ lịch LR 8/16 (lượt này chỉ chạy đến epoch 10), gradient clipping 1.0, deterministic. Evaluate tại epoch 5, 7 và 10; báo checkpoint epoch 10, không chọn theo test AUC.

Padding SAME giữ nguyên số vị trí: trái = floor((k−1)/2), phải = floor(k/2). Kernel chẵn có thêm một vị trí padding bên phải. k=1 không trộn thời gian trong CNN, nhưng LGT vẫn có temporal context. Số tham số Conv tăng theo k nên thí nghiệm thay đổi cả receptive field lẫn số tham số.

Chạy các cell theo thứ tự một lần. Đổi KERNEL_SIZES nếu muốn mở rộng. Cấu hình mặc định gồm 9 lượt × 10 epoch, mỗi lượt evaluate 3 lần.

In [ ]:
import csv
import json
import os
import shlex
import shutil
import subprocess
import sys
from pathlib import Path
import torch
from google.colab import drive

drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/Finetune VadCLIP')
SRC_DIR = PROJECT_ROOT / 'VadCLIP' / 'src'
LIST_DIR = PROJECT_ROOT / 'VadCLIP' / 'list'
FEATURE_ROOT = Path('/content/UCFClipFeatures')
RESULT_ROOT = PROJECT_ROOT / 'code' / 'results'

KERNEL_SIZES = list(range(1, 10))
SCORE_MODE = 'competitive_response_gelu'
SEED = 234
MAX_EPOCH = 10
EVALUATE_EPOCHS = [5, 7, 10]
SWEEP_TAG = 'cnn_k1_9_e10_s234'
RESTART_INCOMPLETE = False  # True explicitly restarts a partial matching run from scratch.
BASELINE_METRICS = None  # Optional path to metrics.json from a self-trained baseline.

if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU runtime first.')
if not SRC_DIR.is_dir():
    raise FileNotFoundError(SRC_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ftfy', 'regex', 'scipy', 'scikit-learn', 'pandas'], check=True)
if not FEATURE_ROOT.is_dir():
    archive = next((PROJECT_ROOT / name for name in ('UCFClipFeatures.zip', 'UCFClipFeatures.tar', 'UCFClipFeatures.tar.gz')
                    if (PROJECT_ROOT / name).is_file()), None)
    if archive is None:
        raise FileNotFoundError('Missing extracted features and feature archive on Drive.')
    local_archive = Path('/content') / archive.name
    if not local_archive.is_file() or local_archive.stat().st_size != archive.stat().st_size:
        shutil.copy2(archive, local_archive)
    command = ['unzip', '-q', str(local_archive), '-d', '/content'] if archive.suffix == '.zip' else ['tar', '-xf', str(local_archive), '-C', '/content']
    subprocess.run(command, check=True)
required = [FEATURE_ROOT]
required += [SRC_DIR / name for name in ('model_cnn_vadclip.py', 'ucf_train_cnn_vadclip.py',
    'ucf_evaluate_cnn_vadclip.py', 'ucf_cnn_common.py', 'test_cnn_kernel_sweep.py')]
required += [LIST_DIR / name for name in ('ucf_CLIP_rgb_relative.csv', 'ucf_CLIP_rgbtest_relative.csv',
    'gt_ucf.npy', 'gt_segment_ucf.npy', 'gt_label_ucf.npy')]
for path in required:
    if not path.exists():
        raise FileNotFoundError(path)
if not KERNEL_SIZES or any(type(k) is not int or k < 1 for k in KERNEL_SIZES) or len(set(KERNEL_SIZES)) != len(KERNEL_SIZES):
    raise ValueError('KERNEL_SIZES must contain distinct positive integers.')
if SCORE_MODE not in ('competitive_response', 'competitive_response_gelu'):
    raise ValueError('Select a competitive response mode.')
if 'validate_cnn_kernel' not in (SRC_DIR / 'model_cnn_vadclip.py').read_text(encoding='utf-8'):
    raise RuntimeError('Upload updated model/train/evaluator files with even-kernel support.')
print('GPU:', torch.cuda.get_device_name(0), '| Kernels:', KERNEL_SIZES, '| Mode:', SCORE_MODE)
print('Runs:', len(KERNEL_SIZES), '| Epochs/run:', MAX_EPOCH)
print('Results:', RESULT_ROOT / 'evaluate' / SWEEP_TAG)


## Train và evaluate tất cả kernel

Cell này kiểm tra cấu hình và chạy tuần tự các lượt. Mỗi lượt có thư mục k1, k2, … riêng; log được in ra console và lưu. Lượt hoàn tất đúng cấu hình sẽ được bỏ qua. Nếu bị ngắt giữa một lượt, mặc định báo lỗi để tránh âm thầm ghi đè; đặt RESTART_INCOMPLETE=True để train lại riêng lượt dở từ scratch, hoặc dùng SWEEP_TAG mới. Không resume optimizer từ checkpoint.

Bảng summary.csv được cập nhật sau từng lượt; epoch_metrics.csv chứa các metric theo epoch. Các lượt trước ở notebook khác không bị ghi đè.

In [ ]:
def command_args(options):
    result = []
    for key, value in options.items():
        if isinstance(value, bool):
            if value:
                result.append('--' + key.replace('_', '-'))
        else:
            result.append('--' + key.replace('_', '-'))
            result.extend(map(str, value if isinstance(value, list) else [value]))
    return result

def run_command(command, log_path=None):
    print('Running:', shlex.join(map(str, command)), flush=True)
    log = None
    if log_path is not None:
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log = log_path.open('a', encoding='utf-8')
        log.write('\nCOMMAND: ' + shlex.join(map(str, command)) + '\n')
    try:
        with subprocess.Popen(command, cwd=SRC_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1, env={**os.environ, 'PYTHONHASHSEED': str(SEED)}) as process:
            for line in process.stdout:
                print(line, end='', flush=True)
                if log:
                    log.write(line)
                    log.flush()
            if process.wait() != 0:
                raise RuntimeError('Command failed; see traceback above. Earlier completed variants are retained.')
    finally:
        if log:
            log.close()

def variant_paths(k):
    name = f'k{k}'
    return {
        'train': RESULT_ROOT / 'train' / SWEEP_TAG / name,
        'eval': RESULT_ROOT / 'evaluate' / SWEEP_TAG / name,
        'analyze': RESULT_ROOT / 'analyze' / SWEEP_TAG / name,
        'epochs': SRC_DIR / 'model' / 'epoch_checkpoints_cnn_single_conv' / SWEEP_TAG / name,
        'final': SRC_DIR / 'model' / 'cnn_single_conv' / SWEEP_TAG / f'{name}.pth',
    }

def train_options(k, paths):
    return {
        'feature_root': str(FEATURE_ROOT), 'train_list': str(LIST_DIR / 'ucf_CLIP_rgb_relative.csv'),
        'cnn_architecture': 'single_conv', 'score_mode': SCORE_MODE, 'kernel_size': k,
        'cnn_channels': 128, 'aggregation': 'topk', 'lambda_alignment': 1.0, 'lambda_text': 1.0,
        'seed': SEED, 'data_seed': SEED, 'deterministic': True, 'grad_clip_norm': 1.0,
        'batch_size': 64, 'lr': 5e-6, 'max_epoch': MAX_EPOCH, 'visual_length': 256,
        'scheduler_milestones': [8, 16], 'scheduler_rate': 0.1,
        'early_stopping_patience': 0, 'num_workers': 2, 'evaluate_epochs': EVALUATE_EPOCHS,
        'test_list': str(LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv'),
        'gt_path': str(LIST_DIR / 'gt_ucf.npy'), 'gt_segment_path': str(LIST_DIR / 'gt_segment_ucf.npy'),
        'gt_label_path': str(LIST_DIR / 'gt_label_ucf.npy'),
        'epoch_checkpoint_dir': str(paths['epochs']), 'output_model_path': str(paths['final']),
        'results_dir': str(paths['train']), 'evaluate_dir': str(paths['eval']), 'analyze_dir': str(paths['analyze']),
    }

def check_config(options, paths):
    config_path = paths['train'] / 'config.json'
    checkpoints_exist = paths['epochs'].exists() and any(paths['epochs'].glob('*.pth'))
    if config_path.is_file():
        saved = json.loads(config_path.read_text(encoding='utf-8'))
        different = [key for key, value in options.items() if saved.get(key) != value]
        if different:
            raise RuntimeError(f'Config mismatch at {different}. Use a new SWEEP_TAG.')
    elif checkpoints_exist or paths['final'].exists():
        raise RuntimeError('Checkpoint exists without config. Use a new SWEEP_TAG.')

def completed(paths):
    summary_path = paths['train'] / 'early_stopping.json'
    final_epoch = paths['epochs'] / f'epoch_{MAX_EPOCH:03d}.pth'
    if not summary_path.is_file():
        return False
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    return final_epoch.is_file() and paths['final'].is_file() and summary['epochs_ran'] == MAX_EPOCH and not summary['stopped_early']

def save_csv(path, rows):
    if not rows:
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

run_command([sys.executable, '-m', 'unittest', 'test_cnn_kernel_sweep'])
# Validate every existing configuration before starting expensive work.
for k in KERNEL_SIZES:
    paths = variant_paths(k)
    check_config(train_options(k, paths), paths)

rows, epoch_rows = [], []
metric_keys = ['classifier_auc', 'classifier_ap', 'alignment_auc', 'alignment_ap', 'ano_auc', 'avg_mAP']
for k in KERNEL_SIZES:
    paths = variant_paths(k)
    options = train_options(k, paths)
    print('\n' + '=' * 70, flush=True)
    print(f'Kernel (512,{k}) | 128 kernels | padding left={(k-1)//2}, right={k//2}', flush=True)
    if completed(paths):
        print('Completed matching run: skip training.', flush=True)
    else:
        partial = paths['epochs'].exists() and any(paths['epochs'].glob('*.pth'))
        if partial and not RESTART_INCOMPLETE:
            raise RuntimeError(f'k{k} is incomplete. Set RESTART_INCOMPLETE=True to restart it from scratch, or use a new SWEEP_TAG.')
        if partial:
            print('Explicitly restarting incomplete matching run from scratch.', flush=True)
        run_command([sys.executable, '-u', str(SRC_DIR / 'ucf_train_cnn_vadclip.py')] + command_args(options),
                    paths['train'] / 'console.log')

    final_eval = paths['eval'] / f'epoch_{MAX_EPOCH:03d}'
    final_analyze = paths['analyze'] / f'epoch_{MAX_EPOCH:03d}'
    if not (final_eval / 'metrics.json').is_file() or not (final_analyze / 'summary.json').is_file():
        eval_options = {
            'model_type': 'cnn', 'model_path': str(paths['epochs'] / f'epoch_{MAX_EPOCH:03d}.pth'),
            'cnn_architecture': 'single_conv', 'score_mode': SCORE_MODE, 'kernel_size': k, 'cnn_channels': 128,
            'train_config_json': str(paths['train'] / 'config.json'), 'feature_root': str(FEATURE_ROOT),
            'test_list': options['test_list'], 'gt_path': options['gt_path'],
            'gt_segment_path': options['gt_segment_path'], 'gt_label_path': options['gt_label_path'],
            'evaluate_dir': str(final_eval), 'analyze_dir': str(final_analyze),
        }
        run_command([sys.executable, '-u', str(SRC_DIR / 'ucf_evaluate_cnn_vadclip.py')] + command_args(eval_options),
                    paths['train'] / 'console.log')
    metrics = json.loads((final_eval / 'metrics.json').read_text(encoding='utf-8'))
    diagnostic = json.loads((final_analyze / 'summary.json').read_text(encoding='utf-8'))
    if metrics['kernel_size'] != k or metrics['score_mode'] != SCORE_MODE:
        raise RuntimeError('Metric/config mismatch; use a clean SWEEP_TAG.')
    rows.append({
        'variant': f'k{k}', 'kernel_size': k, 'channels': 128, 'epoch': MAX_EPOCH,
        'score_mode': SCORE_MODE, 'padding_left': (k-1)//2, 'padding_right': k//2,
        'classifier_parameters': 128*512*k + 128 + 1,
        **{key: metrics[key] for key in metric_keys},
        'top15_anomaly_macro': diagnostic['selected_anomaly_fraction_abnormal_video_macro'],
    })
    history_path = paths['train'] / 'evaluation_history.csv'
    if history_path.is_file():
        with history_path.open(encoding='utf-8', newline='') as handle:
            epoch_rows.extend({'variant': f'k{k}', **row} for row in csv.DictReader(handle))
    save_csv(RESULT_ROOT / 'evaluate' / SWEEP_TAG / 'summary.csv', rows)
    save_csv(RESULT_ROOT / 'evaluate' / SWEEP_TAG / 'epoch_metrics.csv', epoch_rows)
    print('Variant | AUC1 | AP1 | AUC2 | AP2 | Ano-AUC | mAP | Top15 macro')
    for row in rows:
        print(row['variant'] + ' | ' + ' | '.join(f'{row[key]:.2f}' for key in metric_keys)
              + f" | {100*row['top15_anomaly_macro']:.2f}%")
print('Sweep complete:', RESULT_ROOT / 'evaluate' / SWEEP_TAG)


## Đọc lại bảng tổng hợp

Cell này chỉ đọc kết quả đã lưu, không train. Có thể chạy riêng sau khi reconnect Colab và chạy cell setup. Baseline tự train được hiển thị nếu đã điền BASELINE_METRICS; không nạp baseline để khởi tạo các model.

In [ ]:
summary_path = RESULT_ROOT / 'evaluate' / SWEEP_TAG / 'summary.csv'
if not summary_path.is_file():
    raise FileNotFoundError(summary_path)
keys = ['classifier_auc', 'classifier_ap', 'alignment_auc', 'alignment_ap', 'ano_auc', 'avg_mAP']
print('Variant | AUC1 | AP1 | AUC2 | AP2 | Ano-AUC | mAP')
if BASELINE_METRICS is not None:
    baseline = json.loads(Path(BASELINE_METRICS).read_text(encoding='utf-8'))
    print('baseline_self_trained | ' + ' | '.join(f'{float(baseline[key]):.2f}' for key in keys))
with summary_path.open(encoding='utf-8', newline='') as handle:
    for row in csv.DictReader(handle):
        print(row['variant'] + ' | ' + ' | '.join(f'{float(row[key]):.2f}' for key in keys))
print('Summary:', summary_path)
print('Per-epoch:', summary_path.with_name('epoch_metrics.csv'))
